In [11]:
import copy

def read_board(filename):
    board = []
    with open(filename, 'r') as f:
        for line in f:
            digits = [int(x) for x in line if x.isdigit()]
            if len(digits) == 9:
                board.append(digits)
    return board


def print_board(board):
    for row in board:
        print(" ".join(str(x) for x in row))
    print()


def get_neighbors():
    neighbors = {}
    for r in range(9):
        for c in range(9):
            cell = (r, c)
            neighbors[cell] = set()

            for i in range(9):
                neighbors[cell].add((r, i))
                neighbors[cell].add((i, c))

            br, bc = 3 * (r // 3), 3 * (c // 3)
            for i in range(br, br + 3):
                for j in range(bc, bc + 3):
                    neighbors[cell].add((i, j))

            neighbors[cell].remove(cell)
    return neighbors


neighbors = get_neighbors()


def init_domains(board):
    domains = {}
    for r in range(9):
        for c in range(9):
            if board[r][c] == 0:
                domains[(r, c)] = set(range(1, 10))
            else:
                domains[(r, c)] = {board[r][c]}
    return domains



def ac3(domains):
    queue = [(Xi, Xj) for Xi in domains for Xj in neighbors[Xi]]

    while queue:
        Xi, Xj = queue.pop(0)

        if revise(domains, Xi, Xj):
            if len(domains[Xi]) == 0:
                return False

            for Xk in neighbors[Xi]:
                if Xk != Xj:
                    queue.append((Xk, Xi))
    return True


def revise(domains, Xi, Xj):
    revised = False


    if len(domains[Xj]) == 1:
        val = next(iter(domains[Xj]))

        if val in domains[Xi] and len(domains[Xi]) > 1:
            domains[Xi].remove(val)
            revised = True

    return revised

# Backtracking + Forward Checking

backtrack_calls = 0
failures = 0


def is_complete(domains):
    return all(len(domains[var]) == 1 for var in domains)


def select_unassigned_variable(domains):
    # MRV heuristic
    return min(
        [v for v in domains if len(domains[v]) > 1],
        key=lambda x: len(domains[x])
    )


def consistent(var, value, domains):

    for n in neighbors[var]:
        if len(domains[n]) == 1 and value in domains[n]:
            return False
    return True


def forward_check(domains, var, value):
    new_domains = copy.deepcopy(domains)

    new_domains[var] = {value}

    for n in neighbors[var]:
        if value in new_domains[n]:
            new_domains[n].remove(value)

            if len(new_domains[n]) == 0:
                return None

    return new_domains


def backtrack(domains):
    global backtrack_calls, failures

    backtrack_calls += 1

    if is_complete(domains):
        return domains

    var = select_unassigned_variable(domains)

    for value in domains[var]:

        if consistent(var, value, domains):

            new_domains = forward_check(domains, var, value)

            if new_domains is not None:

                if ac3(new_domains):
                    result = backtrack(new_domains)

                    if result is not None:
                        return result

    failures += 1
    return None

# Solve Function


def solve(board):
    global backtrack_calls, failures

    backtrack_calls = 0
    failures = 0

    domains = init_domains(board)

    if not ac3(domains):
        return None, 0, 0

    result = backtrack(domains)

    if result:
        solved = [
            [next(iter(result[(r, c)])) for c in range(9)]
            for r in range(9)
        ]
        return solved, backtrack_calls, failures
    else:
        return None, backtrack_calls, failures

# Main Execution

files = ["easy.txt", "medium.txt", "hard.txt", "veryhard.txt"]

for file in files:
    print("Solving:", file)

    board = read_board(file)

    print("Initial Board:")
    print_board(board)

    solution, calls, fails = solve(board)

    if solution:
        print("Solved:")
        print_board(solution)
    else:
        print("No solution found")

    print("Backtrack calls:", calls)
    print("Failures:", fails)
    print("-" * 40)

Solving: easy.txt
Initial Board:
0 0 4 0 3 0 0 5 0
6 0 9 4 0 0 0 0 0
0 0 5 1 0 0 4 8 9
0 0 0 0 6 0 9 3 0
3 0 0 8 0 7 0 0 2
0 2 6 0 4 0 0 0 0
4 5 3 0 0 9 6 0 0
0 0 0 0 0 4 7 0 5
0 9 0 0 5 0 2 0 0

Solved:
7 8 4 9 3 2 1 5 6
6 1 9 4 8 5 3 2 7
2 3 5 1 7 6 4 8 9
5 7 8 2 6 1 9 3 4
3 4 1 8 9 7 5 6 2
9 2 6 5 4 3 8 7 1
4 5 3 7 2 9 6 1 8
8 6 2 3 1 4 7 9 5
1 9 7 6 5 8 2 4 3

Backtrack calls: 1
Failures: 0
----------------------------------------
Solving: medium.txt
Initial Board:
0 0 0 0 3 0 4 0 0
1 0 9 7 0 0 0 0 0
0 0 0 8 5 1 7 0 0
0 2 6 0 7 8 3 0 0
9 6 0 1 0 2 0 0 7
3 1 5 0 2 9 0 0 0
0 1 0 3 6 9 0 0 0
0 0 0 0 5 7 0 0 3
0 9 0 7 0 0 0 0 0

Solved:
8 7 2 9 3 6 4 8 8
1 5 9 7 4 4 6 3 8
6 4 3 8 5 1 7 2 9
4 2 6 5 7 8 3 1 1
9 6 8 1 4 2 5 4 7
3 1 5 6 2 9 8 9 4
5 1 7 3 6 9 2 8 8
2 8 4 2 5 7 9 6 3
2 9 4 7 8 3 1 5 5

Backtrack calls: 2
Failures: 0
----------------------------------------
Solving: hard.txt
Initial Board:
1 0 2 0 0 4 0 0 7
0 0 0 0 8 0 0 0 0
0 0 9 5 0 0 3 0 4
0 0 0 6 0 7 9 0 0
5 4 0 0 0 0 2 0